# Placeholders in Draft

This notebook answers the placeholders in the current Results draft. It uses the existing project outputs rather than recomputing the access analysis.

Inputs:

- `outputs/tables/fig4_transition_population_by_slr.csv`
- `outputs/tables/fig4_cumulative_population_by_slr.csv`
- `data/processed/analysis/block_level_long_dataset.csv`

Important interpretation: transition-population totals across all SLR scenarios are population-scenario counts. A block's 2020 population can appear in more than one SLR scenario if it remains in an adverse transition category at multiple modeled water levels.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path(r"c:/Users/Vivek/Dropbox/repos/slr_fl_fragile_access"),
    ]
    for candidate in candidates:
        if (candidate / "outputs" / "tables" / "fig4_transition_population_by_slr.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find slr_fl_fragile_access project root.")


PROJECT_ROOT = find_project_root()
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed" / "analysis"

transition_pop = pd.read_csv(TABLES_DIR / "fig4_transition_population_by_slr.csv")
cumulative = pd.read_csv(TABLES_DIR / "fig4_cumulative_population_by_slr.csv")
long_df = pd.read_csv(
    ANALYSIS_DIR / "block_level_long_dataset.csv",
    usecols=["block_geoid", "slr_ft", "baseline_status"],
    dtype={"block_geoid": "string"},
)

SLR_LEVELS = [1, 2, 3, 4, 5, 6]


def pct(numerator, denominator, digits=1):
    return round(100 * numerator / denominator, digits) if denominator else np.nan


def comma(value):
    return f"{int(round(value)):,}"


print(f"Project root: {PROJECT_ROOT}")
print(f"Transition rows: {len(transition_pop):,}")
print(f"Cumulative rows: {len(cumulative):,}")

## Placeholder A: population share in adverse transitions

Draft text:

> Across all SLR scenarios, redundant-to-inundated transitions account for approximately X% of the population in adverse transitions, while transitions that move blocks into fragility or isolation---rather than directly into inundation---account for the remaining X%.

This chunk computes the population-weighted transition shares across all modeled SLR scenarios. The draft sentence needs one small accuracy fix: transitions into fragility or isolation are not literally the entire remaining share, because `fragile to inundated` is another non-redundant-to-inundated pathway.

In [ ]:
all_transition_rows = transition_pop.loc[transition_pop["slr_ft"].isin(SLR_LEVELS)].copy()

total_transition_pop = int(all_transition_rows["pop20"].sum())

redundant_to_inundated_pop = int(
    all_transition_rows.loc[
        all_transition_rows["transition"].eq("Redundant to inundated"),
        "pop20",
    ].sum()
)

into_fragility_or_isolation_labels = [
    "Redundant to fragile",
    "Redundant to isolated",
    "Fragile to isolated",
]
into_fragility_or_isolation_pop = int(
    all_transition_rows.loc[
        all_transition_rows["transition"].isin(into_fragility_or_isolation_labels),
        "pop20",
    ].sum()
)

fragile_to_inundated_pop = int(
    all_transition_rows.loc[
        all_transition_rows["transition"].eq("Fragile to inundated"),
        "pop20",
    ].sum()
)

all_non_redundant_to_inundated_pop = total_transition_pop - redundant_to_inundated_pop

transition_population_summary = pd.DataFrame(
    [
        {
            "category": "Redundant to inundated",
            "population_scenario_count": redundant_to_inundated_pop,
            "share_pct": pct(redundant_to_inundated_pop, total_transition_pop),
        },
        {
            "category": "Into fragility or isolation",
            "population_scenario_count": into_fragility_or_isolation_pop,
            "share_pct": pct(into_fragility_or_isolation_pop, total_transition_pop),
        },
        {
            "category": "Fragile to inundated",
            "population_scenario_count": fragile_to_inundated_pop,
            "share_pct": pct(fragile_to_inundated_pop, total_transition_pop),
        },
        {
            "category": "All non-redundant-to-inundated pathways",
            "population_scenario_count": all_non_redundant_to_inundated_pop,
            "share_pct": pct(all_non_redundant_to_inundated_pop, total_transition_pop),
        },
    ]
)

display(transition_population_summary)

print("Values for placeholder A:")
print(
    f"Redundant-to-inundated = {pct(redundant_to_inundated_pop, total_transition_pop):.1f}% "
    f"of transition-weighted population "
    f"({comma(redundant_to_inundated_pop)} of {comma(total_transition_pop)} population-scenario observations)."
)
print(
    f"All non-redundant-to-inundated pathways = "
    f"{pct(all_non_redundant_to_inundated_pop, total_transition_pop):.1f}%."
)
print(
    f"Transitions into fragility or isolation specifically = "
    f"{pct(into_fragility_or_isolation_pop, total_transition_pop):.1f}%."
)
print(
    f"Fragile-to-inundated, the other remaining pathway, = "
    f"{pct(fragile_to_inundated_pop, total_transition_pop):.1f}%."
)

## Placeholder B: nonlinear growth in total affected population

Draft text:

> The total affected population also grows nonlinearly with SLR, increasing roughly X-fold between 1 ft and 6 ft, with the steepest increases concentrated at the higher scenarios.

This chunk uses `new_fragile_or_worse_pop20`, the cumulative population measure behind the population version of Figure 4b.

In [ ]:
population_growth = cumulative.loc[
    cumulative["slr_ft"].isin(SLR_LEVELS),
    [
        "slr_ft",
        "new_inundated_pop20",
        "new_isolated_or_inundated_pop20",
        "new_fragile_or_worse_pop20",
        "added_by_fragile_pop20",
    ],
].copy()
population_growth["increase_from_previous_scenario"] = population_growth["new_fragile_or_worse_pop20"].diff()
population_growth["fold_vs_1ft"] = (
    population_growth["new_fragile_or_worse_pop20"]
    / population_growth.loc[population_growth["slr_ft"].eq(1), "new_fragile_or_worse_pop20"].iloc[0]
)

pop_1ft = int(population_growth.loc[population_growth["slr_ft"].eq(1), "new_fragile_or_worse_pop20"].iloc[0])
pop_6ft = int(population_growth.loc[population_growth["slr_ft"].eq(6), "new_fragile_or_worse_pop20"].iloc[0])
fold_1_to_6 = pop_6ft / pop_1ft

increments = population_growth.dropna(subset=["increase_from_previous_scenario"]).copy()
steepest_row = increments.loc[increments["increase_from_previous_scenario"].idxmax()]
steepest_to = int(steepest_row["slr_ft"])
steepest_from = steepest_to - 1
steepest_increase = int(steepest_row["increase_from_previous_scenario"])

display(population_growth)

print("Value for placeholder B:")
print(
    f"Affected population grows from {comma(pop_1ft)} at 1 ft to {comma(pop_6ft)} at 6 ft, "
    f"a {fold_1_to_6:.1f}-fold increase."
)
print(
    f"The steepest absolute increase is from {steepest_from} ft to {steepest_to} ft: "
    f"+{comma(steepest_increase)} people."
)

## Placeholder C: baseline fragile share at s = 0

Draft text:

> roughly X% of blocks begin in a fragile state at s = 0

This chunk computes the share among all study-area blocks. It also prints the share among initially accessible blocks only, in case you want that denominator instead.

In [ ]:
baseline = (
    long_df.loc[long_df["slr_ft"].eq(0), ["block_geoid", "baseline_status"]]
    .drop_duplicates("block_geoid")
    .copy()
)

baseline_counts = (
    baseline["baseline_status"]
    .value_counts()
    .rename_axis("baseline_status")
    .reset_index(name="n_blocks")
)
baseline_counts["share_of_all_blocks_pct"] = 100 * baseline_counts["n_blocks"] / baseline_counts["n_blocks"].sum()

n_all_blocks = int(len(baseline))
n_fragile_baseline = int((baseline["baseline_status"] == "fragile").sum())
share_fragile_all = pct(n_fragile_baseline, n_all_blocks)

accessible_baseline = baseline.loc[baseline["baseline_status"].isin(["redundant", "fragile"])]
share_fragile_accessible = pct(n_fragile_baseline, len(accessible_baseline))

display(baseline_counts)

print("Value for placeholder C:")
print(
    f"Baseline fragile share among all blocks = {share_fragile_all:.1f}% "
    f"({comma(n_fragile_baseline)} of {comma(n_all_blocks)} blocks)."
)
print(
    f"If restricted to initially accessible blocks only: {share_fragile_accessible:.1f}% "
    f"({comma(n_fragile_baseline)} of {comma(len(accessible_baseline))} redundant-or-fragile blocks)."
)

## Draft-safe replacement language

This cell prints a paragraph that uses the values above. It slightly revises the first sentence so that the shares sum correctly and do not imply that all non-redundant-to-inundated pathways are fragility/isolation pathways.

In [ ]:
replacement_text = f"""
Across all SLR scenarios, redundant-to-inundated transitions account for approximately
{pct(redundant_to_inundated_pop, total_transition_pop):.1f}\\% of the population in adverse transition pathways.
All other adverse pathways account for the remaining {pct(all_non_redundant_to_inundated_pop, total_transition_pop):.1f}\\%:
{pct(into_fragility_or_isolation_pop, total_transition_pop):.1f}\\% of the transition-weighted population moves into fragility or isolation,
while {pct(fragile_to_inundated_pop, total_transition_pop):.1f}\\% consists of baseline-fragile blocks that become inundated.
The total affected population also grows nonlinearly with SLR, increasing from {comma(pop_1ft)} people at 1 ft
to {comma(pop_6ft)} people at 6 ft, a {fold_1_to_6:.1f}-fold increase. The steepest absolute increase occurs
between {steepest_from} ft and {steepest_to} ft, when the affected population rises by {comma(steepest_increase)} people.

Two sources of variation produce this pattern, and both are invisible to binary metrics of inundation or isolation from SLR.
First, baseline access is already heterogeneous: roughly {share_fragile_all:.1f}\\% of blocks begin in a fragile state at $s = 0$,
dependent on a single dry path before any projected SLR-induced road loss has occurred.
""".strip()

print(replacement_text)